# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. All data entities are uniquely referenced by their `@id` as specified by the Croissant schema.

### Dataset Source
The dataset is accessed using the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed; restart the kernel if needed after installation.
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display key metadata fields
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Citation: {getattr(metadata, 'citeAs', 'N/A')}")

## 2. Data Overview
Review available record sets, the fields in each, and their `@id` references. All navigation will use Croissant `@id` values.

In [ ]:
# List available record sets and fields by their @id.
from pprint import pprint

record_set_ids = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    print('Available Record Sets:')
    for rs in metadata.recordSet:
        rs_id = rs['@id'] if isinstance(rs, dict) else rs
        record_set_ids.append(rs_id)
        print(f'  - {rs_id}')

    # Display fields in the first record set as an example
    print(f"\nFields in first record set ({record_set_ids[0]}):")
    fields = getattr(dataset, 'get_record_set_fields', None)
    if fields:
        field_objs = dataset.get_record_set_fields(record_set_ids[0])
        for f in field_objs:
            print(f"    - {f['@id']} : {f.get('name', '')}")
    else:
        # Generic fallback: try to dump a sample record
        print('No record set field details found via metadata.')
else:
    print('No record sets found in the schema.')

## 3. Data Extraction
Load data from each record set into a pandas DataFrame. We will use the list of `@id` values for each record set. Field and column names match their `@id` as specified in the Croissant metadata, ensuring unique references for each entity.

In [ ]:
# If the metadata explicitly lists record sets, extract them.
# For this FAIR² schema, we must get record_set_ids from previous cell.
if not record_set_ids:
    # One record set is most likely, fallback to trying common IDs.
    # Use the Dataset main @id if recordSet is empty
    record_set_ids = [metadata['@id']] if hasattr(metadata, '__getitem__') and '@id' in metadata else []

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}:\n{df.columns.tolist()}")
        display(df.head())
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")
        continue

## 4. Exploratory Data Analysis (EDA)
Apply basic processing: filtering, normalization, and grouping. All field references use their `@id` values. Update the variable names depending on the fields available (adjust as needed for your specific schema).

In [ ]:
# Choose the main record set and numeric field for demonstration (by @id)
record_set_id = record_set_ids[0]
# Inspect columns to pick a numeric field and optionally a group field:
df = dataframes[record_set_id]
print(f"Record set {record_set_id} has columns: {df.columns.tolist()}")

## Typical numeric fields: 'coverage_percent', 'peptide_count', 'molecular_weight', 'pI' etc. Their @id can be checked from documentation or by dump of df.columns.
## Example: Use the column 'coverage_percent', or choose as present in your schema.
numeric_field = None
candidate_numeric_fields = ['coverage_percent', 'peptide_count', 'molecular_weight', 'pI', 'normalized_abundance']
for col in df.columns:
    if col in candidate_numeric_fields:
        numeric_field = col
        break
if numeric_field is None:
    # Fallback: Use the first numeric dtype column
    numeric_cols = df.select_dtypes(include=['number']).columns
    numeric_field = numeric_cols[0] if len(numeric_cols) else df.columns[0]

print(f"Using numeric field: {numeric_field}")

# Filter records based on threshold
threshold = 10
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (rows: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if available (e.g. protein accession, description, sample id, etc). 
    group_field_candidates = ['sample_id', 'accession', 'description']
    group_field = None
    for g in group_field_candidates:
        if g in df.columns:
            group_field = g
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field}, showing mean {numeric_field} per group:")
        display(grouped_df.head())
else:
    print(f"Field {numeric_field} not found in DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields. All analysis is by field `@id`.

In [ ]:
import matplotlib.pyplot as plt

# Histogram for numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

# Scatter plot: numeric_field vs another numeric field, if present
other_numeric_field = None
for col in df.select_dtypes(include=['number']).columns:
    if col != numeric_field:
        other_numeric_field = col
        break
if other_numeric_field:
    plt.figure(figsize=(6, 4))
    plt.scatter(df[numeric_field], df[other_numeric_field], alpha=0.5)
    plt.xlabel(numeric_field)
    plt.ylabel(other_numeric_field)
    plt.title(f'{numeric_field} vs {other_numeric_field}')
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and analyze the FAIR² mass spectrometry dataset using `mlcroissant`, referencing all columns by their Croissant `@id`. We reviewed metadata, explored record sets, performed basic numeric filtering and normalization, grouped data, and visualized key distributions. For detailed analysis and exact field mappings, always refer to the Croissant dataset documentation and schema.